# **1. Perkenalan Dataset**


Tahap pertama, Anda harus mencari dan menggunakan dataset dengan ketentuan sebagai berikut:

1. **Sumber Dataset**:  
   Dataset dapat diperoleh dari berbagai sumber, seperti public repositories (*Kaggle*, *UCI ML Repository*, *Open Data*) atau data primer yang Anda kumpulkan sendiri.


# **2. Import Library**

Pada tahap ini, Anda perlu mengimpor beberapa pustaka (library) Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning atau deep learning.

In [1]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "dataset" / "raw" / "telco_churn_raw.csv").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "raw" / "telco_churn_raw.csv"
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "preprocessed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


# **3. Memuat Dataset**

Pada tahap ini, Anda perlu memuat dataset ke dalam notebook. Jika dataset dalam format CSV, Anda bisa menggunakan pustaka pandas untuk membacanya. Pastikan untuk mengecek beberapa baris awal dataset untuk memahami strukturnya dan memastikan data telah dimuat dengan benar.

Jika dataset berada di Google Drive, pastikan Anda menghubungkan Google Drive ke Colab terlebih dahulu. Setelah dataset berhasil dimuat, langkah berikutnya adalah memeriksa kesesuaian data dan siap untuk dianalisis lebih lanjut.

Jika dataset berupa unstructured data, silakan sesuaikan dengan format seperti kelas Machine Learning Pengembangan atau Machine Learning Terapan

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset path: {DATA_PATH}")
print(f"Shape: {df.shape}")
display(df.head())


Dataset path: /home/kuzan/Documents/SMSL/Eksperimen_SML_kuzanf3b/dataset/raw/telco_churn_raw.csv
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini, Anda akan melakukan **Exploratory Data Analysis (EDA)** untuk memahami karakteristik dataset.

Tujuan dari EDA adalah untuk memperoleh wawasan awal yang mendalam mengenai data dan menentukan langkah selanjutnya dalam analisis atau pemodelan.

In [3]:
# Pemeriksaan awal kualitas data dan struktur kolom.
print("Data types:")
display(df.dtypes.rename("dtype").to_frame())

print("Missing values:")
display(df.isna().sum().rename("missing").to_frame())

print("Blank TotalCharges values:", (df["TotalCharges"].astype(str).str.strip() == "").sum())
print("Duplicate rows:", df.duplicated().sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
churn_counts = df["Churn"].value_counts().reindex(["No", "Yes"])
axes[0].bar(churn_counts.index, churn_counts.values, color=["tab:blue", "tab:orange"])
axes[0].set_title("Distribusi target Churn")
axes[0].set_xlabel("Churn")
axes[0].set_ylabel("Jumlah pelanggan")
axes[1].hist(df["MonthlyCharges"], bins=30, color="tab:blue")
axes[1].set_title("Distribusi MonthlyCharges")
axes[1].set_xlabel("MonthlyCharges")
plt.close(fig)

print("Target proportion:")
display(df["Churn"].value_counts(normalize=True).rename("proportion").to_frame())
display(df[["tenure", "MonthlyCharges"]].describe().T)


Data types:


,dtype
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


Missing values:


,missing
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


Blank TotalCharges values: 11
Duplicate rows: 0
Target proportion:


,proportion
Churn,
No,0.73463
Yes,0.26537


,count,mean,std,min,25%,50%,75%,max
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

## Rationale preprocessing

- `customerID` dihapus karena hanya identifier dan tidak membawa sinyal generalisasi.
- `TotalCharges` dikonversi ke numerik; nilai kosong menjadi missing value dan diimputasi dengan median train.
- Target `Churn` dipetakan dari `No/Yes` menjadi `0/1`.
- Data dibagi secara stratified sebelum `fit` preprocessing. Imputasi, encoding, dan scaling hanya belajar dari train untuk mencegah data leakage.
- Kolom kategorikal memakai `handle_unknown="ignore"` agar transformasi tetap aman ketika kategori baru muncul.


In [4]:
# 1. Membersihkan tipe data dan identifier.
prepared = df.copy()
prepared["TotalCharges"] = pd.to_numeric(
    prepared["TotalCharges"].astype(str).str.strip(), errors="coerce"
)
prepared = prepared.drop_duplicates().reset_index(drop=True)
prepared = prepared.drop(columns=["customerID"])
prepared["Churn"] = prepared["Churn"].map({"No": 0, "Yes": 1}).astype("int64")

# 2. Memisahkan target sebelum split agar evaluasi tidak mengalami leakage.
X = prepared.drop(columns=["Churn"])
y = prepared["Churn"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# 3. Fitting imputasi, encoding, dan scaling hanya pada data train.
numeric_features = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
categorical_features = [
    column for column in X.columns if column not in numeric_features
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()
X_train_processed = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

# Simpan hasil eksplorasi agar dapat diperiksa sebelum otomatisasi pada task berikutnya.
train_output = X_train_processed.copy()
train_output["Churn"] = y_train
test_output = X_test_processed.copy()
test_output["Churn"] = y_test
train_output.to_csv(OUTPUT_DIR / "telco_churn_train.csv", index=False)
test_output.to_csv(OUTPUT_DIR / "telco_churn_test.csv", index=False)

print(f"Train shape: {X_train_processed.shape}")
print(f"Test shape: {X_test_processed.shape}")
print(f"Missing values after preprocessing: {int(X_train_processed.isna().sum().sum())}")
display(X_train_processed.head())


Train shape: (5634, 45)
Test shape: (1409, 45)
Missing values after preprocessing: 0


/tmp/ipykernel_12713/2176399726.py:3: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  prepared["TotalCharges"] = pd.to_numeric(
/tmp/ipykernel_12713/2176399726.py:8: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
Y

,numeric__SeniorCitizen,numeric__tenure,numeric__MonthlyCharges,numeric__TotalCharges,categorical__gender_Female,categorical__gender_Male,categorical__Partner_No,categorical__Partner_Yes,categorical__Dependents_No,categorical__Dependents_Yes,...,categorical__StreamingMovies_Yes,categorical__Contract_Month-to-month,categorical__Contract_One year,categorical__Contract_Two year,categorical__PaperlessBilling_No,categorical__PaperlessBilling_Yes,categorical__PaymentMethod_Bank transfer (automatic),categorical__PaymentMethod_Credit card (automatic),categorical__PaymentMethod_Electronic check,categorical__PaymentMethod_Mailed check
3738,-0.441773,0.102371,-0.521976,-0.263290,0.0,1.0,1.0,0.0,1.0,0.0,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
3151,-0.441773,-0.711743,0.337478,-0.504815,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
4860,-0.441773,-0.793155,-0.809013,-0.751214,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
3867,-0.441773,-0.263980,0.284384,-0.173700,1.0,0.0,0.0,1.0,1.0,0.0,...,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0
3810,-0.441773,-1.281624,-0.676279,-0.990851,0.0,1.0,0.0,1.0,0.0,1.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
